In [ ]:
import ee
import geemap
from math import sqrt
from utils.variables import (
    PROJECT,
    TEST_SITE_IDs,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    GLC_CLASSES,
    NFW_THRESHOLD,
    MAX_EDGE_DIST,
    OPENING_RADIUS,
    MAX_PATCH_SIZE,
)

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Data imports
PAs = ee.FeatureCollection("WCMC/WDPA/current/polygons")
OECMs = ee.FeatureCollection("WCMC/WDOECM/current/polygons")
GLC = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
HGFC = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")
NFW = ee.ImageCollection(
    "projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection"
)

In [ ]:
# Get selected test sites
def get_test_sites():
    # Combine terrestrial PAs and OECMs into one collection
    all_sites = (
        ee.FeatureCollection([PAs, OECMs])
        .flatten()
        .filter(ee.Filter.eq("REALM", "Terrestrial"))
    )
    # Select test sites by ID
    test_sites = all_sites.filter(ee.Filter.inList("SITE_ID", TEST_SITE_IDs))
    return test_sites


test_sites = get_test_sites()

In [ ]:
# Select site
site_id = 352159

In [ ]:
# Get site geometry
def get_site_geom(site_id):
    return test_sites.filter(ee.Filter.eq("SITE_ID", site_id)).geometry()


site_geom = get_site_geom(site_id)

In [ ]:
# Define START_YR
# (If a PA was designated after the start of the analysis period, use the designation year as the start year)
def set_start_yr():
    designation_yr = (
        test_sites.filter(ee.Filter.eq("SITE_ID", site_id))
        .first()
        .get("STATUS_YR")
        .getInfo()
    )
    return max(ANALYSIS_START_YR, designation_yr)


START_YR = set_start_yr()

In [ ]:
# Process Global Land Cover Change data
def process_GLC():
    # Mosaic images in the collection that intersect test sites
    GLC_mosaic = GLC.filterBounds(test_sites).mosaic()
    # Select and rename bands corresponding to analysis period
    analysis_years = list[int](range(START_YR, ANALYSIS_END_YR + 1))
    band_names = [f"b{year - 2000 + 1}" for year in analysis_years]
    new_band_names = [f"GLC_{year}" for year in analysis_years]
    GLC_selected = GLC_mosaic.select(band_names, new_band_names)

    # Remap class values to 1-36
    def remap_classes(band):
        return (
            GLC_selected.select(band)
            .remap(GLC_CLASSES, ee.List.sequence(1, len(GLC_CLASSES)), defaultValue=0)
            .rename([band])
        )

    remapped_bands = [remap_classes(band) for band in new_band_names]
    GLC_remapped = ee.Image.cat(remapped_bands)
    return GLC_remapped


GLC_processed = process_GLC()

In [ ]:
# Process Global Pasture Watch data
def process_GPW():
    # Filter GPW collection to analysis period
    year_strings = [str(year) for year in range(START_YR, ANALYSIS_END_YR + 1)]
    GPW_filtered = GPW.filter(ee.Filter.inList("system:index", year_strings)).toBands()
    GPW_renamed = GPW_filtered.rename([f"GPW_{year}" for year in year_strings])
    return GPW_renamed.unmask()


GPW_processed = process_GPW()

In [ ]:
# Process Natural Forests of the World data
def process_NFW():
    # Mosaic images in the collection that intersect test sites
    NFW_mosaic = NFW.filterBounds(test_sites).mosaic()
    # Set probability threshold
    NFW_thresholded = NFW_mosaic.gte(NFW_THRESHOLD)
    return NFW_thresholded


NFW_processed = process_NFW()

In [ ]:
# Process Hansen Global Forest Change data
def process_HGFC():
    # Select forest loss band
    HGFC_selected = HGFC.select("lossyear")
    # Mask to forest loss within analysis period
    analysis_mask = HGFC_selected.gte(START_YR - 2000).And(
        HGFC_selected.lte(ANALYSIS_END_YR - 2000)
    )
    HGFC_masked = HGFC_selected.updateMask(analysis_mask)
    return HGFC_masked.unmask()


HGFC_processed = process_HGFC()

In [ ]:
# Create habitat raster for ANALYSIS_END_YR by creating and inverting a non-habitat mask
def get_habitat_raster():
    # Select GLC data for ANALYSIS_END_YR
    GLC_current = GLC_processed.select(f"GLC_{ANALYSIS_END_YR}")
    # Select cropland (1-4) and impervious surfaces (30)
    anthro_classes = ee.List([1, 2, 3, 4, 30])
    anthro_mask = GLC_current.remap(
        anthro_classes, ee.List.repeat(1, anthro_classes.size()), defaultValue=0
    )
    # Select any forest loss pixels during the analysis period
    forest_loss_mask = HGFC_processed.gt(0)
    # Select any cultivated grassland pixels using GPW
    GPW_current = GPW_processed.select(f"GPW_{ANALYSIS_END_YR}")
    pasture_mask = GPW_current.eq(1)
    # Select any pixels of "Forest" class that do not coincide with Natural Forests
    closed_forest_classes = ee.List([6, 8, 10, 12, 14])
    planted_forest_mask = (
        GLC_current.remap(
            closed_forest_classes,
            ee.List.repeat(1, closed_forest_classes.size()),
            defaultValue=0,
        ).updateMask(NFW_processed.eq(0))
    ).unmask(0)
    # Combine all non-habitat masks and invert; apply mask to landcover raster
    non_habitat_mask = (
        anthro_mask.add(forest_loss_mask).add(pasture_mask).add(planted_forest_mask)
    )
    habitat_mask = non_habitat_mask.eq(0)
    habitat_raster = GLC_current.updateMask(habitat_mask)
    return habitat_raster


habitat_raster = get_habitat_raster()

In [ ]:
# Calculate Habitat Extent score for ANALYSIS_END_YR within a PA
def calc_habitat_extent_score(site_geom):
    # Calculate total area of site
    site_area = site_geom.area().getInfo()
    # Calculate total area of habitat within site
    habitat_area = (
        ee.Image.pixelArea()
        .updateMask(habitat_raster)
        .reduceRegion(ee.Reducer.sum(), site_geom, scale=30, maxPixels=1e13)
        .get("area")
        .getInfo()
    )
    habitat_proportion = habitat_area / site_area
    return habitat_proportion


habitat_extent_score = calc_habitat_extent_score(site_geom)
# print(habitat_extent_score)

In [ ]:
# Create edge distance raster (distance to nearest non-habitat pixel)
def get_edge_distance_raster(site_geom):
    # Buffer site to max edge distance
    site_buffered = site_geom.buffer(MAX_EDGE_DIST + OPENING_RADIUS)
    # Create and invert habitat binary (habitat = 0, non-habitat = 1)
    habitat_binary = habitat_raster.clip(site_buffered).unmask().eq(0)
    # Morphologically open (erode and dilate) the habitat binary to remove tiny areas of non-habitat
    habitat_binary_opened = habitat_binary.focalMin(
        radius=OPENING_RADIUS, kernelType="square", units="meters"
    ).focalMax(radius=OPENING_RADIUS, kernelType="square", units="meters")
    # Calculate distance to nearest non-habitat pixel
    edge_distance = habitat_binary_opened.distance(
        ee.Kernel.euclidean(MAX_EDGE_DIST, units="meters")
    )
    # Clean edge distance layer
    edge_distance_cleaned = (
        edge_distance.min(MAX_EDGE_DIST)  # Cap values to saturate at max edge distance
        .unmask(MAX_EDGE_DIST)  # Set masked values (values beyond the kernel) to max
        .clip(site_geom)  # Clip to PA
        .selfMask()  # Mask all 0 (non-habitat) pixels
        .rename("edge_distance")
    )  # Rename band for clarity
    return edge_distance_cleaned


edge_distance_raster = get_edge_distance_raster(site_geom)

In [ ]:
# Create patch size raster (number of connected habitat pixels)
def get_patch_size_raster(site_geom):
    patch_size = (
        habitat_raster.gt(0)  # Make habitat binary
        .connectedPixelCount(maxSize=1024, eightConnected=False)
        .min(MAX_PATCH_SIZE * 1000000)  # Cap values at max patch size
        .rename("patch_size")
        .clip(site_geom)
    )
    return patch_size


patch_size_raster = get_patch_size_raster(site_geom)
patch_scale = sqrt((MAX_PATCH_SIZE * 1000000) / 1024)

In [ ]:
# Calculate Habitat Intactness score using edge distance and patch size
def calc_intactness_score(site_geom):
    avg_edge_distance = (
        edge_distance_raster.reduceRegion(
            ee.Reducer.mean(), site_geom, scale=30, maxPixels=1e13
        )
        .get("edge_distance")
        .getInfo()
    )
    avg_edge_distance_scaled = avg_edge_distance / MAX_EDGE_DIST
    avg_patch_size = (
        patch_size_raster.reduceRegion(
            ee.Reducer.mean(), site_geom, scale=patch_scale, maxPixels=1e13
        )
        .get("patch_size")
        .getInfo()
    )
    avg_patch_size_scaled = avg_patch_size / (MAX_PATCH_SIZE * 1000000)
    intactness_score = sqrt(
        avg_edge_distance_scaled * avg_patch_size_scaled
    )  # Geometric mean
    return intactness_score


habitat_intactness_score = calc_intactness_score(site_geom)
# print(habitat_intactness_score)

In [ ]:
# Calculate overall Habitat Condition score using Habitat Extent and Habitat Intactness
def calc_habitat_condition_score():
    return habitat_extent_score * habitat_intactness_score


habitat_condition_score = calc_habitat_condition_score()
# print(habitat_condition_score)

In [ ]:
# Create Habitat Loss raster (binary of habitat loss over analysis period)
def get_habitat_loss_raster():
    # Create start year habitat binary
    GLC_start = GLC_processed.select(f"GLC_{START_YR}")
    GPW_start = GPW_processed.select(f"GPW_{START_YR}")
    lc_start = GLC_start.where(GPW_start.eq(1), 37)
    anthro_classes = ee.List([1, 2, 3, 4, 30, 37])
    habitat_start = lc_start.remap(
        anthro_classes, ee.List.repeat(0, anthro_classes.size()), defaultValue=1
    )

    # Create end year anthro binary
    forest_loss_binary = HGFC_processed.gt(0)
    GLC_end = GLC_processed.select(f"GLC_{ANALYSIS_END_YR}")
    GPW_end = GPW_processed.select(f"GPW_{ANALYSIS_END_YR}")
    lc_end = GLC_end.where(GPW_end.eq(1), 37)
    anthro_end = lc_end.remap(
        anthro_classes, ee.List.repeat(1, anthro_classes.size()), defaultValue=0
    ).where(forest_loss_binary, 1)

    # Create habitat loss binary
    habitat_loss_binary = habitat_start.And(anthro_end)
    return habitat_loss_binary


habitat_loss_raster = get_habitat_loss_raster()

In [ ]:
# Calculate Habitat Loss score as the proportion of habitat protected
def calc_habitat_loss_score(site_geom):
    habitat_loss_area = (
        ee.Image.pixelArea()
        .updateMask(habitat_loss_raster)
        .reduceRegion(ee.Reducer.sum(), site_geom, scale=30, maxPixels=1e13)
        .get("area")
        .getInfo()
    )
    site_area = site_geom.area().getInfo()
    habitat_loss_proportion = habitat_loss_area / site_area
    habitat_loss_score = 1 - habitat_loss_proportion
    return habitat_loss_score


habitat_loss_score = calc_habitat_loss_score(site_geom)
print(habitat_loss_score)

In [ ]:
# Helper function for calculating class area and percent area
def calc_class_area_and_pct(class_image, site_geom, top_n=4):
    """
    Calculate area and percent area for each class in a classified image.

    Args:
        class_image: ee.Image with a single band containing class values
        site_geom: ee.Geometry of the site
        top_n: Integer to filter to top N classes by area (default: 4)

    Returns:
        ee.Dictionary with keys like '{class}_area' and '{class}_pct'
    """
    # Get class name from the image band name
    class_name = class_image.bandNames().get(0)

    # Calculate area by class
    area_by_class = (
        ee.Image.pixelArea()
        .divide(1000000)  # Get pixel area in sq km
        .addBands(class_image)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(
                groupField=1,
                groupName=class_name,
            ),
            geometry=site_geom,
            scale=30,
            maxPixels=1e13,
        )
    )

    # Convert list to dictionary
    def dict_from_list(item, acc):
        item = ee.Dictionary(item)
        key = item.get(class_name)
        value = item.get("sum")
        return ee.Dictionary(acc).set(key, value)

    class_dict = ee.Dictionary(
        ee.List(area_by_class.get("groups")).iterate(dict_from_list, ee.Dictionary({}))
    )

    # Filter to top N classes by area
    class_dict = class_dict.select(
        class_dict.keys().sort(class_dict.values()).reverse().slice(0, top_n)
    )

    # Append '_area' suffix to keys
    new_keys = class_dict.keys().map(lambda key: ee.String(key).cat("_area"))
    class_dict = ee.Dictionary.fromLists(new_keys, class_dict.values())

    # Add percentage calculations
    site_area = site_geom.area().divide(1000000).getInfo()

    def add_pct_area(key, value):
        pct_key = ee.String(key).slice(0, -5).cat("_pct")
        pct_value = ee.Number(value).divide(site_area).multiply(100)
        return ee.Dictionary().set(key, value).set(pct_key, pct_value)

    def combine_dicts(item, acc):
        return ee.Dictionary(acc).combine(ee.Dictionary(item), True)

    result_dict = ee.Dictionary(
        class_dict.map(add_pct_area).values().iterate(combine_dicts, ee.Dictionary({}))
    )

    return result_dict

In [ ]:
# Calculate area and percent area for each of the 4 drivers of habitat conversion
def get_driver_class_image():
    GLC_end = GLC_processed.select(f"GLC_{ANALYSIS_END_YR}")
    GPW_end = GPW_processed.select(f"GPW_{ANALYSIS_END_YR}")
    lc_end = GLC_end.where(GPW_end.eq(1), 37)
    # Remap landcover classes to driver categories:
    # 1-4 -> 1 (cropland)
    # 30 -> 2 (built-up)
    # 37 -> 3 (pasture)
    # everything else -> 4 (deforestation without conversion)
    driver_class = (
        lc_end.updateMask(habitat_loss_raster)
        .remap([1, 2, 3, 4, 30, 37], [1, 1, 1, 1, 2, 3], defaultValue=4)
        .rename("driver_class")
    )
    return driver_class


driver_class = get_driver_class_image()
driver_dict = calc_class_area_and_pct(driver_class, site_geom)

print(driver_dict.getInfo())

In [ ]:
# Calculate area and percent area for each of the top 4 types of habitat lost
def get_habitat_class_image():
    GLC_start = GLC_processed.select(f"GLC_{START_YR}")
    habitat_class = GLC_start.updateMask(habitat_loss_raster).rename("habitat_class")
    return habitat_class


habitat_class = get_habitat_class_image()
habitat_dict = calc_class_area_and_pct(habitat_class, site_geom)

print(habitat_dict.getInfo())

In [ ]:
# Visualization

from utils.variables import GLC_PALETTE

Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")

# Imagery for Jaci-Paraná
ls_medians = ee.ImageCollection("projects/skytruth-tech/assets/TPA/BRA_testPAs_scaled")
pre_image = ls_medians.filter(ee.Filter.eq("year", START_YR))
post_image = ls_medians.filter(ee.Filter.eq("year", ANALYSIS_END_YR))
Map.addLayer(
    pre_image,
    {"bands": ["R", "G", "B"], "min": 0, "max": 3500},
    "Landsat Median, start year",
)
Map.addLayer(
    post_image,
    {"bands": ["R", "G", "B"], "min": 0, "max": 3500},
    "Landsat Median, end year",
)

# Map.addLayer(site_geom, {"color": "red"}, "Test site")
# Map.addLayer(GLC_processed.select('GLC_2022'), {'min':0, 'max':36, 'palette': GLC_PALETTE}, "Global Land Cover")
# Map.addLayer(GPW_processed.select('GPW_2022').selfMask(), {'min':1, 'max':2, 'palette': ['#ffcd73','#ff9916']}, "Global Pasture Watch")
# Map.addLayer(NFW_processed.selfMask(), {'palette': 'teal'}, "Natural Forests of the World")
# Map.addLayer(HGFC_processed.selfMask(), {'min':START_YR-2000, 'max':ANALYSIS_END_YR-2000, 'palette': ['yellow', 'red']}, "Hansen Global Forest Change")
# Map.addLayer(test_sites, {"color": "red"}, "Test sites")
# Map.addLayer(habitat_raster.clip(site_geom), {'palette': GLC_PALETTE}, "Habitat extent")
# Map.addLayer(edge_distance_raster, {'min':0, 'max':MAX_EDGE_DIST, 'palette': ['red', 'white']}, 'Edge distance')
# Map.addLayer(patch_size_raster.reproject(crs=habitat_raster.projection(), scale=patch_scale), {'min':1, 'max':1024, 'palette': ['red', 'white']}, 'Patch size')
# Map.addLayer(habitat_loss_raster.selfMask(), {'palette': ['red']}, "Habitat loss (binary)")
Map.addLayer(
    driver_class.selfMask(),
    {"min": 1, "max": 4, "palette": ["yellow", "red", "orange", "green"]},
    "Drivers of habitat loss",
)
Map.addLayer(
    habitat_class.selfMask(), {"palette": GLC_PALETTE}, "Types of habitat lost"
)

Map.centerObject(site_geom)

Map